In [ ]:
import sys
from pathlib import Path

# Ensure repo root is on the path so anp_emulator and train_anp_emulator are importable
sys.path.insert(0, str(Path('..').resolve()))


In [ ]:
from pathlib import Path
from types import SimpleNamespace
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

import zuko.flows

from train_anp_emulator import (
    ALL_PROFILE_LOG_TARGETS,
    ALL_PROFILE_TARGETS,
    MeanModel,
    build_tasks,
    split_tasks,
    resolve_profile_file,
    load_theta_table,
    predict_mean_from_raw_x,
    get_mean_model_config,
)
from train_nf_emulator import ConditionalNF

In [ ]:
# ── Load NF checkpoint ──
#
# Set RUN_DIR to the NF training run directory.
# It should contain best_model.pt (or an epoch_*.pt checkpoint).

RUN_DIR = Path('nf_training_runs/nf_all_profiles_XXXXXXXXXX')  # <-- update with actual run
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

if not RUN_DIR.exists():
    raise FileNotFoundError(f'Run directory not found: {RUN_DIR}')

# Find best checkpoint
ckpt_path = RUN_DIR / 'best_model.pt'
if not ckpt_path.exists():
    # Fall back to latest epoch checkpoint
    epoch_ckpts = sorted(RUN_DIR.glob('epoch_*.pt'))
    if not epoch_ckpts:
        raise FileNotFoundError(f'No checkpoint found in {RUN_DIR}')
    ckpt_path = epoch_ckpts[-1]

ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
print(f'Loaded checkpoint: {ckpt_path.name}')
print(f'  Keys: {sorted(ckpt.keys())}')

In [ ]:
# ── Reconstruct NF model and normalization from checkpoint ──

model_config = ckpt['model_config']
nf_args = ckpt['args']
norm_np = ckpt['norm']
target_names = ckpt['target_names']

# Build model
nf_model = ConditionalNF(
    x_dim=model_config['x_dim'],
    y_dim=model_config['y_dim'],
    hidden_features=model_config['hidden_features'],
    n_transforms=model_config['n_transforms'],
    n_bins=model_config['n_bins'],
    dropout=model_config.get('dropout', 0.0),
)
nf_model.load_state_dict(ckpt['model_state_dict'])
nf_model.to(DEVICE)
nf_model.eval()

# Normalization tensors
x_mean = torch.tensor(norm_np['x_mean'], dtype=torch.float32, device=DEVICE)
x_std  = torch.tensor(norm_np['x_std'],  dtype=torch.float32, device=DEVICE)
y_mean = torch.tensor(norm_np['y_mean'], dtype=torch.float32, device=DEVICE)
y_std  = torch.tensor(norm_np['y_std'],  dtype=torch.float32, device=DEVICE)

# Mean model (if present)
mean_model = None
mean_prior_enabled = ckpt.get('mean_prior_enabled', False)
if mean_prior_enabled and ckpt.get('mean_model_state_dict') is not None:
    mm_config = ckpt['mean_model_config']
    mean_model = MeanModel(
        y_dim=mm_config['y_dim'],
        hidden_dim=mm_config['hidden_dim'],
        use_redshift=mm_config['use_redshift'],
        theta_dim=mm_config.get('theta_dim', 0),
        theta_start_idx=mm_config.get('theta_start_idx', 2),
        n_hidden_layers=mm_config.get('n_hidden_layers', 2),
    )
    mean_model.load_state_dict(ckpt['mean_model_state_dict'])
    mean_model.to(DEVICE)
    mean_model.eval()
    print(f'Mean model loaded: {sum(p.numel() for p in mean_model.parameters())} params')
else:
    print('No mean model — NF trained on direct targets.')

# Metadata
x_dim = model_config['x_dim']
y_dim = model_config['y_dim']
theta_dim = int(nf_args.get('theta_dim', 35))
theta_start_idx = int(nf_args.get('theta_start_idx', 2))
use_continuous_redshift = bool(nf_args.get('use_continuous_redshift_feature', True))
redshift_feature_idx = int(nf_args.get('redshift_feature_idx', theta_start_idx + theta_dim))

# Log-space channel indices (for converting predictions back to physical units)
log_channel_indices = [i for i, name in enumerate(target_names) if name in ALL_PROFILE_LOG_TARGETS]

print(f'NF model: x_dim={x_dim}, y_dim={y_dim}')
print(f'Target names: {target_names}')
print(f'Log-space channels: {[target_names[i] for i in log_channel_indices]}')
print(f'Mean prior: {mean_prior_enabled}')
print(f'Params: {sum(p.numel() for p in nf_model.parameters()):,}')

In [ ]:
# ── NF predict function ──
# Replicates the ANP Emulator.predict() pipeline for the NF model.

@torch.no_grad()
def nf_predict(
    theta: np.ndarray,
    M: np.ndarray,
    r_bins: np.ndarray,
    snapnum: int,
    redshift: float,
    n_samples: int = 30,
    fields: list = None,
):
    """Predict profiles with the NF model.

    Parameters
    ----------
    theta : (theta_dim,) fiducial parameter vector
    M : (n_halo,) halo masses in M_sun
    r_bins : (n_halo, n_r) radial bins as r/R500
    snapnum : int
    redshift : float
    n_samples : number of flow samples for mean/std estimation
    fields : list of field names to return (None = all target_names)

    Returns
    -------
    dict with keys 'mean', 'std', 'field_names'
        mean: (n_halo, n_r, n_fields) in physical units
        std:  (n_halo, n_r, n_fields) in physical units
    """
    theta_use = np.asarray(theta, dtype=np.float32)
    if theta_use.ndim == 1:
        theta_use = np.tile(theta_use, (len(M), 1))  # (n_halo, theta_dim)

    n_halo = len(M)
    n_r = r_bins.shape[1] if r_bins.ndim == 2 else r_bins.shape[0]

    if r_bins.ndim == 1:
        r_bins = np.tile(r_bins, (n_halo, 1))

    log_m = np.log10(np.clip(M, 1e10, None)).astype(np.float32)
    log_r = np.log10(np.clip(r_bins, 1e-4, None)).astype(np.float32)

    # Build raw x: (n_halo, n_r, x_dim)
    x_raw = np.zeros((n_halo, n_r, x_dim), dtype=np.float32)
    x_raw[..., 0] = log_m[:, None]
    x_raw[..., 1] = log_r
    x_raw[..., theta_start_idx:theta_start_idx + theta_dim] = theta_use[:, None, :]
    if use_continuous_redshift and redshift_feature_idx < x_dim:
        x_raw[..., redshift_feature_idx] = float(redshift)

    # Flatten to (N, x_dim) and normalise
    x_flat = x_raw.reshape(-1, x_dim)
    x_norm = (x_flat - x_mean.cpu().numpy()) / x_std.cpu().numpy()
    x_t = torch.tensor(x_norm, dtype=torch.float32, device=DEVICE)

    # Sample from flow
    BATCH = 8192
    N = x_t.shape[0]
    mu_parts, std_parts = [], []
    for i in range(0, N, BATCH):
        xb = x_t[i:i+BATCH]
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(DEVICE == 'cuda')):
            samples = nf_model.sample(xb, n_samples=n_samples)  # (n_samples, B, y_dim)
        mu_parts.append(samples.float().mean(dim=0).cpu())
        std_parts.append(samples.float().std(dim=0).cpu())

    mu_norm = torch.cat(mu_parts, dim=0)  # (N, y_dim)
    std_norm = torch.cat(std_parts, dim=0)

    # De-normalise: y = mu_norm * y_std + y_mean  (residual space)
    mu_resid = mu_norm * y_std.cpu() + y_mean.cpu()
    std_resid = std_norm * y_std.cpu()

    # Add mean model back if present
    if mean_model is not None:
        x_raw_t = torch.tensor(x_flat, dtype=torch.float32, device=DEVICE)
        mm_parts = []
        for i in range(0, N, BATCH):
            xb_raw = x_raw_t[i:i+BATCH]
            mm_pred = predict_mean_from_raw_x(xb_raw, mean_model)
            mm_parts.append(mm_pred.cpu())
        mm_contrib = torch.cat(mm_parts, dim=0)  # (N, y_dim)
        mu_phys = mu_resid + mm_contrib
    else:
        mu_phys = mu_resid

    # Convert log-space channels to physical units: 10^x
    mu_phys_np = mu_phys.numpy()
    std_phys_np = std_resid.numpy()
    ln10 = math.log(10.0)
    for idx in log_channel_indices:
        phys_val = np.power(10.0, mu_phys_np[:, idx])
        std_phys_np[:, idx] = ln10 * phys_val * np.abs(std_phys_np[:, idx])
        mu_phys_np[:, idx] = phys_val

    # Reshape back to (n_halo, n_r, y_dim)
    mu_out = mu_phys_np.reshape(n_halo, n_r, y_dim)
    std_out = std_phys_np.reshape(n_halo, n_r, y_dim)

    # Filter fields if requested
    out_field_names = list(target_names)
    if fields is not None:
        field_idx = [target_names.index(f) for f in fields if f in target_names]
        mu_out = mu_out[:, :, field_idx]
        std_out = std_out[:, :, field_idx]
        out_field_names = [target_names[i] for i in field_idx]

    return {
        'mean': mu_out,
        'std': std_out,
        'field_names': out_field_names,
    }

print('nf_predict() defined.')

In [ ]:
# ── CV-seed validation setup across all snapshots/redshifts ──
import re

CV_PROFILE_DIR = Path('/mnt/home/mlee1/ceph/Profiles_cy_test/')
CV_FIELDS = ['pressure', 'temperature', 'gas_density']
CV_MASS_CUT = 1e12
CV_NSAMPLES = 30

if not CV_PROFILE_DIR.exists():
    raise FileNotFoundError(f'Profile directory not found: {CV_PROFILE_DIR}')


def _parse_snapshot_redshifts_text(mapping_text):
    out = {}
    for chunk in str(mapping_text).split(','):
        chunk = chunk.strip()
        if not chunk:
            continue
        snap_str, sep, z_str = chunk.partition(':')
        if sep != ':':
            continue
        out[int(snap_str.strip())] = float(z_str.strip())
    return out


def _resolve_training_snaps_and_redshifts(run_args):
    snaps = []
    if isinstance(run_args.get('resolved_snapnums', None), (list, tuple)):
        snaps = [int(s) for s in run_args['resolved_snapnums']]
    elif isinstance(run_args.get('snapnums', None), (list, tuple)):
        snaps = [int(s) for s in run_args['snapnums']]
    elif run_args.get('snapnum', None) is not None:
        snaps = [int(run_args['snapnum'])]
    snaps = sorted(set(snaps))

    redshift_by_snap = {}
    if isinstance(run_args.get('redshift_by_snap', None), dict):
        redshift_by_snap = {int(k): float(v) for k, v in run_args['redshift_by_snap'].items()}
    elif isinstance(run_args.get('snapshot_redshifts', None), str):
        redshift_by_snap = _parse_snapshot_redshifts_text(run_args['snapshot_redshifts'])
    for s in snaps:
        redshift_by_snap.setdefault(int(s), np.nan)
    return snaps, redshift_by_snap


training_snapnums, training_redshift_by_snap = _resolve_training_snaps_and_redshifts(nf_args)
if len(training_snapnums) == 0:
    raise RuntimeError('Could not determine training snapshots from checkpoint args')

# Build map: snapnum -> {CV_tag -> file_path}
cv_file_pattern = re.compile(r'IllustrisTNG_(CV_\d+)_snap(\d+)\.npz$')
cv_tag_to_file_by_snap = {}
for fp in sorted(CV_PROFILE_DIR.glob('IllustrisTNG_CV_*_snap*.npz')):
    m = cv_file_pattern.match(fp.name)
    if m is None:
        continue
    tag = m.group(1)
    snap = int(m.group(2))
    if snap not in training_snapnums:
        continue
    cv_tag_to_file_by_snap.setdefault(snap, {})[tag] = fp

cv_snaps_available = sorted(cv_tag_to_file_by_snap.keys())
if len(cv_snaps_available) == 0:
    raise FileNotFoundError(
        f'No CV files found for training snapshots in {CV_PROFILE_DIR}. '
        f'Training snapshots: {training_snapnums}'
    )

# Resolve fiducial theta.
oneP_param_csv = Path('/mnt/home/mlee1/Sims/IllustrisTNG/L50n512/1P/CosmoAstroSeed_IllustrisTNG_L50n512_1P.txt')
if not oneP_param_csv.exists():
    raise FileNotFoundError(f'1P parameter file not found: {oneP_param_csv}')
oneP_theta_df = pd.read_csv(oneP_param_csv, sep=r'\s+', engine='python')
oneP_theta_df = oneP_theta_df.rename(columns={'#Name': 'tag'})
theta_fid = oneP_theta_df[oneP_theta_df['tag'] == '1P_p1_0'].iloc[0, 1:-1].to_numpy(dtype=np.float32)

cv_snap_summary_rows = []
for snap in cv_snaps_available:
    cv_snap_summary_rows.append({
        'snapnum': int(snap),
        'redshift': float(training_redshift_by_snap.get(int(snap), np.nan)),
        'n_cv_runs': int(len(cv_tag_to_file_by_snap[snap])),
    })

cv_snap_summary_df = pd.DataFrame(cv_snap_summary_rows).sort_values('snapnum').reset_index(drop=True)
print(f'Training snapshots from checkpoint: {training_snapnums}')
print(f'CV snapshots with files found: {cv_snaps_available}')
print('theta_fid shape:', theta_fid.shape)
display(cv_snap_summary_df)

In [ ]:
# ── Run NF predictions on each CV simulation for each snapshot ──
cv_predictions_by_snap = {}
cv_summary_rows = []

for snap in cv_snaps_available:
    cv_predictions = {}
    z_snap = float(training_redshift_by_snap.get(int(snap), np.nan))

    for tag, fp in sorted(cv_tag_to_file_by_snap[snap].items()):
        with np.load(fp) as dat:
            masses = np.asarray(dat['M500c'], dtype=np.float64)
            r500c = np.asarray(dat['R500c'], dtype=np.float64)
            radial_bins = np.asarray(dat['radial_bins'], dtype=np.float64)

            true_profiles = {}
            for fld in CV_FIELDS:
                key = f'{fld}_array'
                if key not in dat:
                    raise KeyError(f'Missing {key} in {fp.name}')
                true_profiles[fld] = np.asarray(dat[key], dtype=np.float64)

        n_halo = masses.shape[0]
        n_r = radial_bins.shape[0]

        # Per-halo r/R500 grid
        r_bins_rr500 = (
            radial_bins[None, :]
            / np.maximum(r500c[:, None], 1e-12)
        ).astype(np.float32)

        pred = nf_predict(
            theta=theta_fid,
            M=masses.astype(np.float32),
            r_bins=r_bins_rr500,
            snapnum=int(snap),
            redshift=float(z_snap),
            n_samples=CV_NSAMPLES,
            fields=CV_FIELDS,
        )

        pred_mu = {}
        pred_sd = {}
        for j, fld in enumerate(pred['field_names']):
            pred_mu[fld] = np.asarray(pred['mean'][:, :, j], dtype=np.float64)
            pred_sd[fld] = np.asarray(pred['std'][:, :, j], dtype=np.float64)

        cv_predictions[tag] = {
            'file': str(fp),
            'snapnum': int(snap),
            'redshift': float(z_snap),
            'M500c': masses,
            'R500c': r500c,
            'rr500': r_bins_rr500,
            'radial_bins': radial_bins,
            'true_profiles': true_profiles,
            'pred_mu': pred_mu,
            'pred_sd': pred_sd,
        }

        cv_summary_rows.append({
            'snapnum': int(snap),
            'redshift': float(z_snap),
            'tag': tag,
            'n_halos': int(n_halo),
            'n_r': int(n_r),
            'm500c_min': float(np.min(masses)),
            'm500c_median': float(np.median(masses)),
            'm500c_max': float(np.max(masses)),
        })

    cv_predictions_by_snap[int(snap)] = cv_predictions

cv_summary_df = pd.DataFrame(cv_summary_rows).sort_values(['snapnum', 'tag']).reset_index(drop=True)
first_snap = int(cv_snaps_available[0])
cv_predictions = cv_predictions_by_snap[first_snap]

print(f'Built cv_predictions_by_snap for {len(cv_predictions_by_snap)} snapshots.')
display(cv_summary_df.head(20))

In [ ]:
# ── Visual check: scatter capture grid ──

def plot_cv_scatter_capture_grid(fields=('pressure', 'temperature', 'gas_density'),
                                 mass_bin='high', snapnum=None):
    if snapnum is None:
        snap = int(sorted(cv_predictions_by_snap.keys())[0])
    else:
        snap = int(snapnum)
    cvp = cv_predictions_by_snap[snap]
    tags = sorted(cvp.keys())
    z = float(training_redshift_by_snap.get(int(snap), np.nan))
    radial_bins = cvp[tags[0]]['radial_bins']

    fig, axes = plt.subplots(2, len(fields), figsize=(5 * len(fields), 8),
                             constrained_layout=True)
    if len(fields) == 1:
        axes = np.array(axes).reshape(2, 1)

    for col, field in enumerate(fields):
        t_medians, p_medians = [], []
        for tag in tags:
            rec = cvp[tag]
            mass = rec['M500c']
            if mass_bin == 'low':
                mask = mass < CV_MASS_CUT
            elif mass_bin == 'high':
                mask = mass >= CV_MASS_CUT
            else:
                mask = np.ones_like(mass, dtype=bool)
            if np.sum(mask) == 0:
                continue
            true_arr = np.clip(rec['true_profiles'][field][mask], 1e-30, None)
            pred_arr = np.clip(rec['pred_mu'][field][mask], 1e-30, None)
            t_medians.append(np.median(true_arr, axis=0))
            p_medians.append(np.median(pred_arr, axis=0))

        ax_top = axes[0, col]
        ax_bot = axes[1, col]

        if len(t_medians) < 2:
            ax_top.text(0.5, 0.5, 'Need >=2 CV runs', ha='center', va='center',
                        transform=ax_top.transAxes)
            ax_bot.text(0.5, 0.5, 'Need >=2 CV runs', ha='center', va='center',
                        transform=ax_bot.transAxes)
            ax_top.set_title(f'{field} | {mass_bin} mass')
            continue

        t_medians = np.asarray(t_medians)
        p_medians = np.asarray(p_medians)
        t_mean = np.mean(t_medians, axis=0)
        p_mean = np.mean(p_medians, axis=0)
        t_scatter = np.std(np.log10(t_medians), axis=0, ddof=1)
        p_scatter = np.std(np.log10(p_medians), axis=0, ddof=1)

        for i in range(t_medians.shape[0]):
            ax_top.plot(radial_bins, t_medians[i], color='tab:blue', alpha=0.2, lw=1.0)
            ax_top.plot(radial_bins, p_medians[i], color='tab:orange', alpha=0.2, lw=1.0)
        ax_top.plot(radial_bins, t_mean, color='tab:blue', lw=2.0, label='true mean of CV medians')
        ax_top.plot(radial_bins, p_mean, color='tab:orange', lw=2.0, ls='--',
                    label='NF mean of CV medians')
        ax_top.set_xscale('log'); ax_top.set_yscale('log')
        ax_top.set_xlabel('radius [kpc]'); ax_top.set_ylabel(field)
        ax_top.set_title(f'{field} | {mass_bin} mass')
        if col == 0:
            ax_top.legend(fontsize=8)

        ax_bot.plot(radial_bins, t_scatter, color='tab:blue', lw=2.0,
                    label='true CV scatter (std log10)')
        ax_bot.plot(radial_bins, p_scatter, color='tab:orange', lw=2.0, ls='--',
                    label='NF CV scatter (std log10)')
        ax_bot.set_xscale('log')
        ax_bot.set_xlabel('radius [kpc]')
        ax_bot.set_ylabel('std(log10 median profile)')
        ax_bot.set_title(f'{field} scatter capture')
        if col == 0:
            ax_bot.legend(fontsize=8)

    fig.suptitle(f'NF CV scatter capture | snap={snap} | z={z:.3f}', y=1.02)
    plt.show()


for _snap in cv_snaps_available:
    plot_cv_scatter_capture_grid(
        fields=('pressure', 'temperature', 'gas_density'),
        mass_bin='high', snapnum=_snap,
    )

In [ ]:
# ── Mass-bin median-profile validation across all halos, per snapshot ──

def plot_cv_massbin_median_match(
    fields=('pressure', 'temperature', 'gas_density'),
    logm_start=12.0,
    n_mass_bins=6,
    min_count_per_bin=20,
    eps=1e-30,
):
    from matplotlib.lines import Line2D

    for snap in cv_snaps_available:
        cvp = cv_predictions_by_snap[snap]
        z_snap = training_redshift_by_snap.get(snap, np.nan)
        tags = sorted(cvp.keys())
        radial_bins = cvp[tags[0]]['radial_bins']

        # Pool masses across all CV runs
        all_logm = np.concatenate([
            np.log10(np.clip(cvp[t]['M500c'], 1e-30, None)) for t in tags
        ])
        pool = all_logm[all_logm >= logm_start]
        if len(pool) < min_count_per_bin * 2:
            print(f'snap {snap}: too few high-mass halos ({len(pool)}) — skipping.')
            continue
        edges = np.unique(np.quantile(pool, np.linspace(0, 1, n_mass_bins + 1)))
        n_bins = len(edges) - 1

        fig, axes = plt.subplots(
            2 * n_bins, len(fields),
            figsize=(5 * len(fields), 2.8 * 2 * n_bins),
            constrained_layout=True,
            gridspec_kw={'height_ratios': [3, 1.2] * n_bins},
        )

        for i_bin in range(n_bins):
            lo, hi = edges[i_bin], edges[i_bin + 1]
            for col, fld in enumerate(fields):
                ax_main = axes[2 * i_bin, col]
                ax_res  = axes[2 * i_bin + 1, col]

                true_stack, pred_stack = [], []
                for tag in tags:
                    lgm = np.log10(np.clip(cvp[tag]['M500c'], 1e-30, None))
                    mask = (lgm >= lo) & (lgm <= hi) if i_bin == n_bins - 1 else (lgm >= lo) & (lgm < hi)
                    if not mask.any():
                        continue
                    true_stack.append(cvp[tag]['true_profiles'][fld][mask])
                    pred_stack.append(cvp[tag]['pred_mu'][fld][mask])

                if not true_stack:
                    ax_main.text(0.5, 0.5, 'No halos', ha='center', va='center',
                                 transform=ax_main.transAxes)
                    continue

                true_all = np.vstack(true_stack)
                pred_all = np.vstack(pred_stack)
                true_med = np.median(true_all, axis=0)
                pred_med = np.median(pred_all, axis=0)

                ax_main.plot(radial_bins, np.clip(true_med, eps, None), 'k-', lw=2.5, alpha=0.85)
                ax_main.plot(radial_bins, np.clip(pred_med, eps, None),
                             color='tab:orange', lw=2.0, ls='--', alpha=0.9)
                ax_main.set_xscale('log'); ax_main.set_yscale('log')
                if i_bin == 0:
                    ax_main.set_title(fld, fontsize=12)
                if col == 0:
                    ax_main.set_ylabel(
                        f'$10^{{{lo:.1f}}}$–$10^{{{hi:.1f}}}$ $M_\\odot$',
                        fontsize=10,
                    )

                denom = np.clip(true_med, eps, None)
                res = np.log10(np.clip(pred_med, eps, None)) - np.log10(denom)
                ax_res.plot(radial_bins, res, color='tab:orange', lw=1.8, ls='--')
                ax_res.axhline(0.0, color='k', lw=0.7, alpha=0.5)
                ax_res.set_xscale('log'); ax_res.set_ylim(-0.5, 0.5)
                if col == 0:
                    ax_res.set_ylabel(r'$\Delta$log$_{10}$', fontsize=10)
                if i_bin == n_bins - 1:
                    ax_res.set_xlabel('radius [kpc]', fontsize=11)

        legend_handles = [
            Line2D([0], [0], color='k', lw=2.5, label='truth (CV median)'),
            Line2D([0], [0], color='tab:orange', lw=2, ls='--', label='NF (median)'),
        ]
        fig.legend(handles=legend_handles, loc='upper center', ncol=2,
                   frameon=False, bbox_to_anchor=(0.5, 1.04), fontsize=11)
        fig.suptitle(
            f'NF CV mass-bin median profiles | snap={snap} z={z_snap:.1f}',
            y=1.07, fontsize=13,
        )
        plt.show()


plot_cv_massbin_median_match()

## Ideal Gas Law Self-Consistency

For physical profiles: $\log_{10}(C) = \log_{10}(P) - \log_{10}(\rho) - \log_{10}(T)$  
Compare how well NF-emulated profiles satisfy this relation vs the true CV profiles.

In [ ]:
# Ideal gas law: log10(P) = log10(rho) + log10(T) + log10(C)
_ig_fields = ('gas_density', 'temperature', 'pressure')
_ig_ok = all(f in CV_FIELDS for f in _ig_fields)

if not _ig_ok:
    print(f'Skipped — need gas_density, temperature, pressure in CV_FIELDS={CV_FIELDS}')
else:
    n_snaps = len(cv_snaps_available)
    fig, axes = plt.subplots(n_snaps, 2, figsize=(14, 4.5 * n_snaps),
                             constrained_layout=True, squeeze=False)

    for i_snap, snap in enumerate(cv_snaps_available):
        z_snap = training_redshift_by_snap.get(snap, np.nan)
        preds_snap = cv_predictions_by_snap[snap]
        all_logC_true, all_logC_pred, all_rr500 = [], [], []

        for tag, entry in sorted(preds_snap.items()):
            tp = entry['true_profiles']
            pm = entry['pred_mu']
            rr = entry['rr500']
            logC_true = (np.log10(np.clip(tp['pressure'], 1e-30, None))
                         - np.log10(np.clip(tp['gas_density'], 1e-30, None))
                         - np.log10(np.clip(tp['temperature'], 1e-30, None)))
            logC_pred = (np.log10(np.clip(pm['pressure'], 1e-30, None))
                         - np.log10(np.clip(pm['gas_density'], 1e-30, None))
                         - np.log10(np.clip(pm['temperature'], 1e-30, None)))
            masses = entry['M500c']
            mask_mass = masses >= CV_MASS_CUT
            if mask_mass.sum() == 0:
                continue
            all_logC_true.append(logC_true[mask_mass])
            all_logC_pred.append(logC_pred[mask_mass])
            all_rr500.append(rr[mask_mass])

        if len(all_logC_true) == 0:
            axes[i_snap, 0].text(0.5, 0.5, 'No data', transform=axes[i_snap, 0].transAxes,
                                 ha='center', va='center')
            continue

        logC_true_all = np.concatenate(all_logC_true, axis=0)
        logC_pred_all = np.concatenate(all_logC_pred, axis=0)
        rr500_all = np.concatenate(all_rr500, axis=0)
        rr_common = np.median(rr500_all, axis=0)

        ax_rad = axes[i_snap, 0]
        for label, logC_arr, color, ls in [
            ('Truth', logC_true_all, 'C0', '-'),
            ('NF', logC_pred_all, 'C3', '--'),
        ]:
            med = np.median(logC_arr, axis=0)
            lo = np.percentile(logC_arr, 16, axis=0)
            hi = np.percentile(logC_arr, 84, axis=0)
            ax_rad.plot(rr_common, med, ls, color=color, lw=1.8, label=label)
            ax_rad.fill_between(rr_common, lo, hi, color=color, alpha=0.15)

        ax_rad.axhline(4.146, ls=':', color='grey', lw=1, label='Expected (4.146)')
        ax_rad.set_xscale('log')
        ax_rad.set_xlabel('r / R$_{500}$')
        ax_rad.set_ylabel(r'log$_{10}$(C) = log$_{10}$(P) $-$ log$_{10}$(\rho) $-$ log$_{10}$(T)')
        ax_rad.set_title(f'snap {snap}  z={z_snap:.1f}  (M$_{{500}}$ ≥ {CV_MASS_CUT:.0e})')
        ax_rad.legend(fontsize=9)
        ax_rad.grid(True, alpha=0.3)

        delta = (logC_pred_all - logC_true_all).ravel()
        delta = delta[np.isfinite(delta)]
        ax_hist = axes[i_snap, 1]
        ax_hist.hist(delta, bins=80, density=True, alpha=0.7, color='C4', edgecolor='k', lw=0.3)
        ax_hist.axvline(0, ls='-', color='k', lw=0.8)
        ax_hist.axvline(np.median(delta), ls='--', color='C3', lw=1.5,
                        label=f'median={np.median(delta):.4f}')
        ax_hist.set_xlabel(r'$\Delta$ log$_{10}$(C)  [pred $-$ true]')
        ax_hist.set_ylabel('Density')
        ax_hist.set_title(
            f'Ideal gas violation  (σ={np.std(delta):.4f} dex, '
            f'|Δ|$_{{50}}$={np.median(np.abs(delta)):.4f})'
        )
        ax_hist.legend(fontsize=9)
        ax_hist.grid(True, alpha=0.3)

    fig.suptitle('Ideal Gas Law Self-Consistency: NF vs Truth (CV set)',
                 fontweight='bold', fontsize=14, y=1.01)
    plt.show()

## Per-Radius Error Diagnostics

1. Per-radius RMSE profiles — where does accuracy degrade?
2. Bias vs scatter decomposition
3. Intrinsic scatter vs model error

In [ ]:
# ── Per-radius RMSE (log10 space) for CV predictions ──

DIAG_FIELDS = ['gas_density', 'temperature', 'pressure']
DIAG_SNAP = cv_snaps_available[0]
eps = 1e-30

cvp = cv_predictions_by_snap[DIAG_SNAP]
tags = sorted(cvp.keys())
radial_bins = cvp[tags[0]]['radial_bins']
n_r = len(radial_bins)

log10_resid_sq = {fld: [] for fld in DIAG_FIELDS}
log10_bias     = {fld: [] for fld in DIAG_FIELDS}

for tag in tags:
    rec = cvp[tag]
    mass = rec['M500c']
    mask = mass >= CV_MASS_CUT
    for fld in DIAG_FIELDS:
        true_arr = np.log10(np.clip(rec['true_profiles'][fld][mask], eps, None))
        pred_arr = np.log10(np.clip(rec['pred_mu'][fld][mask], eps, None))
        resid = pred_arr - true_arr
        log10_resid_sq[fld].append(resid ** 2)
        log10_bias[fld].append(resid)

fig, axes = plt.subplots(2, len(DIAG_FIELDS), figsize=(5 * len(DIAG_FIELDS), 8),
                          constrained_layout=True, sharex=True)

core_boundary = int(np.ceil(0.2 * n_r))

for col, fld in enumerate(DIAG_FIELDS):
    all_sq = np.vstack(log10_resid_sq[fld])
    all_bias = np.vstack(log10_bias[fld])
    rmse_per_r = np.sqrt(np.mean(all_sq, axis=0))
    bias_per_r = np.mean(all_bias, axis=0)
    scatter_per_r = np.std(all_bias, axis=0)

    ax = axes[0, col]
    ax.plot(radial_bins, rmse_per_r, 'k-', lw=2, label='RMSE')
    ax.axvline(radial_bins[core_boundary], ls=':', color='red', alpha=0.6,
               label=f'core boundary (bin {core_boundary})')
    ax.set_xscale('log'); ax.set_ylabel('RMSE [dex]')
    ax.set_title(f'{fld}'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax2 = axes[1, col]
    ax2.plot(radial_bins, bias_per_r, 'C3-', lw=2, label='bias (mean residual)')
    ax2.fill_between(radial_bins, bias_per_r - scatter_per_r, bias_per_r + scatter_per_r,
                      color='C3', alpha=0.2, label=r'$\pm 1\sigma$ scatter')
    ax2.axhline(0, color='k', lw=0.8, alpha=0.5)
    ax2.axvline(radial_bins[core_boundary], ls=':', color='red', alpha=0.6)
    ax2.set_xscale('log')
    ax2.set_xlabel('radius [kpc]')
    ax2.set_ylabel(r'$\log_{10}(\mathrm{pred}) - \log_{10}(\mathrm{true})$ [dex]')
    ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

z_snap = training_redshift_by_snap.get(DIAG_SNAP, np.nan)
fig.suptitle(
    f'NF per-radius error decomposition | snap={DIAG_SNAP} z={z_snap:.1f} | M500 >= {CV_MASS_CUT:.0e}',
    y=1.02,
)
plt.show()

# Summary table: core vs outer RMSE
core_idx = int(np.ceil(0.2 * n_r))
print(f'\nCore = first {core_idx} bins (r < {radial_bins[core_idx]:.1f} kpc)')
print(f'Outer = remaining {n_r - core_idx} bins\n')
for fld in DIAG_FIELDS:
    all_sq = np.vstack(log10_resid_sq[fld])
    rmse_core = np.sqrt(np.mean(all_sq[:, :core_idx]))
    rmse_outer = np.sqrt(np.mean(all_sq[:, core_idx:]))
    rmse_all = np.sqrt(np.mean(all_sq))
    print(f'{fld:15s}  RMSE_core={rmse_core:.4f}  RMSE_outer={rmse_outer:.4f}  '
          f'RMSE_all={rmse_all:.4f}  ratio(core/outer)={rmse_core/max(rmse_outer,1e-10):.2f}x')

In [ ]:
# ── Intrinsic scatter vs model error per radius ──

fig, axes = plt.subplots(1, len(DIAG_FIELDS), figsize=(5 * len(DIAG_FIELDS), 4),
                          constrained_layout=True)
if len(DIAG_FIELDS) == 1:
    axes = [axes]

for col, fld in enumerate(DIAG_FIELDS):
    true_log_all = []
    for tag in tags:
        rec = cvp[tag]
        mask = rec['M500c'] >= CV_MASS_CUT
        if mask.sum() == 0:
            continue
        true_log_all.append(np.log10(np.clip(rec['true_profiles'][fld][mask], eps, None)))

    true_log_all = np.vstack(true_log_all)
    intrinsic_scatter = np.std(true_log_all, axis=0)
    all_sq = np.vstack(log10_resid_sq[fld])
    model_rmse = np.sqrt(np.mean(all_sq, axis=0))

    ax = axes[col]
    ax.plot(radial_bins, intrinsic_scatter, 'C0-', lw=2, label='intrinsic scatter (std of true)')
    ax.plot(radial_bins, model_rmse, 'C3--', lw=2, label='NF RMSE')
    ax.plot(radial_bins, model_rmse / np.clip(intrinsic_scatter, 1e-6, None),
            'C2:', lw=1.5, label='RMSE / scatter')
    ax.axvline(radial_bins[core_boundary], ls=':', color='red', alpha=0.6)
    ax.set_xscale('log')
    ax.set_xlabel('radius [kpc]')
    ax.set_ylabel('[dex] or ratio')
    ax.set_title(f'{fld}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('NF: Intrinsic scatter vs model error per radius', y=1.02)
plt.show()

for fld in DIAG_FIELDS:
    true_log_all = np.vstack([
        np.log10(np.clip(cvp[t]['true_profiles'][fld][cvp[t]['M500c'] >= CV_MASS_CUT], eps, None))
        for t in tags if (cvp[t]['M500c'] >= CV_MASS_CUT).sum() > 0
    ])
    scatter_core = np.mean(np.std(true_log_all[:, :core_idx], axis=0))
    scatter_outer = np.mean(np.std(true_log_all[:, core_idx:], axis=0))
    all_sq = np.vstack(log10_resid_sq[fld])
    rmse_core = np.sqrt(np.mean(all_sq[:, :core_idx]))
    rmse_outer = np.sqrt(np.mean(all_sq[:, core_idx:]))
    print(f'{fld:15s}  core: scatter={scatter_core:.4f} RMSE={rmse_core:.4f} '
          f'RMSE/scatter={rmse_core/max(scatter_core,1e-6):.2f}  |  '
          f'outer: scatter={scatter_outer:.4f} RMSE={rmse_outer:.4f} '
          f'RMSE/scatter={rmse_outer/max(scatter_outer,1e-6):.2f}')

In [ ]:
# ── Per-halo core residual distribution ──

fig, axes = plt.subplots(1, len(DIAG_FIELDS), figsize=(5 * len(DIAG_FIELDS), 5),
                          constrained_layout=True)
if len(DIAG_FIELDS) == 1:
    axes = [axes]

for col, fld in enumerate(DIAG_FIELDS):
    halo_masses_list, halo_core_rmse_list = [], []
    for tag in tags:
        rec = cvp[tag]
        mass = rec['M500c']
        mask = mass >= CV_MASS_CUT
        if mask.sum() == 0:
            continue
        true_log = np.log10(np.clip(rec['true_profiles'][fld][mask], eps, None))
        pred_log = np.log10(np.clip(rec['pred_mu'][fld][mask], eps, None))
        resid = pred_log - true_log
        core_rmse = np.sqrt(np.mean(resid[:, :core_boundary] ** 2, axis=1))
        halo_masses_list.append(mass[mask])
        halo_core_rmse_list.append(core_rmse)

    all_m = np.concatenate(halo_masses_list)
    all_core_rmse = np.concatenate(halo_core_rmse_list)

    ax = axes[col]
    ax.scatter(np.log10(all_m), all_core_rmse, s=6, alpha=0.4, c='C0', rasterized=True)
    log_m_bins = np.linspace(np.log10(all_m.min()), np.log10(all_m.max()), 12)
    bin_centers = 0.5 * (log_m_bins[:-1] + log_m_bins[1:])
    bin_medians = []
    for i in range(len(log_m_bins) - 1):
        in_bin = (np.log10(all_m) >= log_m_bins[i]) & (np.log10(all_m) < log_m_bins[i+1])
        bin_medians.append(np.median(all_core_rmse[in_bin]) if in_bin.sum() > 5 else np.nan)
    ax.plot(bin_centers, bin_medians, 'C3-o', lw=2, ms=5, label='median in mass bin')
    ax.set_xlabel(r'$\log_{10}(M_{500c})$')
    ax.set_ylabel('Core RMSE [dex]')
    ax.set_title(f'{fld}')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

fig.suptitle(
    f'NF per-halo core RMSE vs mass | snap={DIAG_SNAP} | core = first {core_boundary} bins',
    y=1.02,
)
plt.show()

print(f'Total high-mass halos across all CV seeds: {len(all_m)}')
for fld in DIAG_FIELDS:
    core_rmse_all = np.concatenate([
        np.sqrt(np.mean(
            (np.log10(np.clip(cvp[t]['pred_mu'][fld][cvp[t]['M500c'] >= CV_MASS_CUT], eps, None))
             - np.log10(np.clip(cvp[t]['true_profiles'][fld][cvp[t]['M500c'] >= CV_MASS_CUT], eps, None)))[:, :core_boundary] ** 2, axis=1))
        for t in tags if (cvp[t]['M500c'] >= CV_MASS_CUT).sum() > 0
    ])
    pcts = np.percentile(core_rmse_all, [25, 50, 75, 90, 95])
    print(f'{fld:15s}  core RMSE percentiles: p25={pcts[0]:.4f} p50={pcts[1]:.4f} '
          f'p75={pcts[2]:.4f} p90={pcts[3]:.4f} p95={pcts[4]:.4f}')

In [ ]:
# ── Worst-case halo core profiles ──

n_worst = 6

for fld in DIAG_FIELDS:
    halo_records = []
    for tag in tags:
        rec = cvp[tag]
        mass = rec['M500c']
        mask = mass >= CV_MASS_CUT
        if mask.sum() == 0:
            continue
        true_log = np.log10(np.clip(rec['true_profiles'][fld][mask], eps, None))
        pred_log = np.log10(np.clip(rec['pred_mu'][fld][mask], eps, None))
        resid_core = (pred_log - true_log)[:, :core_boundary]
        core_rmse = np.sqrt(np.mean(resid_core ** 2, axis=1))
        true_phys = rec['true_profiles'][fld][mask]
        pred_phys = rec['pred_mu'][fld][mask]
        m_sel = mass[mask]
        for ih in range(len(core_rmse)):
            halo_records.append({
                'tag': tag, 'halo_idx': ih, 'core_rmse': core_rmse[ih],
                'mass': m_sel[ih], 'true': true_phys[ih], 'pred': pred_phys[ih],
            })

    halo_records.sort(key=lambda x: x['core_rmse'], reverse=True)
    worst = halo_records[:n_worst]

    fig, axes = plt.subplots(1, n_worst, figsize=(3.5 * n_worst, 3.5),
                              constrained_layout=True, sharey=True)
    for i, rec in enumerate(worst):
        ax = axes[i]
        ax.plot(radial_bins, rec['true'], 'C0-', lw=1.5, label='true')
        ax.plot(radial_bins, rec['pred'], 'C3--', lw=1.5, label='NF pred')
        ax.axvline(radial_bins[core_boundary], ls=':', color='red', alpha=0.5)
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel('r [kpc]')
        ax.set_title(f'{rec["tag"]} h{rec["halo_idx"]}\n'
                     f'M={rec["mass"]:.2e}\ncore RMSE={rec["core_rmse"]:.3f} dex',
                     fontsize=8)
        if i == 0:
            ax.set_ylabel(fld); ax.legend(fontsize=7)

    fig.suptitle(f'NF: {fld}: {n_worst} worst core halos', y=1.04)
    plt.show()

## Training History

In [ ]:
# ── Training loss history ──
import json

history_path = RUN_DIR / 'history.json'
if not history_path.exists():
    print(f'No history.json found at {history_path}')
else:
    with open(history_path) as f:
        history_raw = json.load(f)

    if isinstance(history_raw, list):
        history = {}
        for rec in history_raw:
            for k, v in rec.items():
                history.setdefault(k, []).append(v)
    else:
        history = history_raw

    epochs = history.get('epoch', list(range(1, len(history.get('train_nll', [])) + 1)))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)

    # Panel 1: NLL
    ax = axes[0]
    if 'train_nll' in history:
        ax.plot(epochs, history['train_nll'], 'C0-', alpha=0.5, lw=1, label='train NLL')
    if 'val_nll' in history:
        ax.plot(epochs, history['val_nll'], 'C3-', lw=1.5, label='val NLL')
    ax.set_xlabel('epoch'); ax.set_ylabel('NLL')
    ax.set_title('Negative log-likelihood')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # Panel 2: RMSE (if logged)
    ax = axes[1]
    has_rmse = False
    if 'val_rmse_original_units' in history:
        vals = [v for v in history['val_rmse_original_units'] if v is not None]
        if vals:
            ep_filtered = [e for e, v in zip(epochs, history['val_rmse_original_units']) if v is not None]
            ax.plot(ep_filtered, vals, 'C0-', lw=1.5, label='val RMSE (orig units)')
            has_rmse = True
    if has_rmse:
        ax.set_xlabel('epoch'); ax.set_ylabel('RMSE')
        ax.set_title('Validation RMSE (original units)')
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'No RMSE logged', ha='center', va='center',
                transform=ax.transAxes)

    # Panel 3: Per-target RMSE
    ax = axes[2]
    has_per_target = False
    for tname in target_names:
        key = f'val_{tname}_rmse_orig'
        if key in history:
            vals = [v for v in history[key] if v is not None]
            if vals:
                ep_f = [e for e, v in zip(epochs, history[key]) if v is not None]
                ax.plot(ep_f, vals, lw=1.2, label=tname, alpha=0.8)
                has_per_target = True
    if has_per_target:
        ax.set_xlabel('epoch'); ax.set_ylabel('RMSE (orig units)')
        ax.set_title('Per-target val RMSE')
        ax.legend(fontsize=7, ncol=2); ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'No per-target RMSE logged', ha='center', va='center',
                transform=ax.transAxes)

    # Panel 4: LR schedule
    if 'lr' in history:
        fig2, ax_lr = plt.subplots(figsize=(8, 3))
        ax_lr.plot(epochs, history['lr'], 'C2-', lw=1.5)
        ax_lr.set_xlabel('epoch'); ax_lr.set_ylabel('learning rate')
        ax_lr.set_title('Learning rate schedule')
        ax_lr.grid(True, alpha=0.3)
        fig2.tight_layout()
        plt.show()

    fig.suptitle(f'NF Training History: {RUN_DIR.name}', y=1.02)
    plt.show()

    print('Available history keys:', sorted(history.keys()))

## NF Log-Likelihood Evaluation on CV Set

Unlike the ANP, the NF directly provides calibrated log-likelihoods.  
Below we evaluate $\log p(y_{\mathrm{true}} | x)$ on the CV data — a metric unique to the NF.

In [ ]:
# ── Per-field NLL on CV set (normalised space) ──
#
# This evaluates the flow's density estimate on held-out CV profiles.
# Lower NLL = better density model.

@torch.no_grad()
def nf_logprob_cv(
    theta: np.ndarray,
    M: np.ndarray,
    r_bins: np.ndarray,
    true_profiles: dict,
    snapnum: int,
    redshift: float,
) -> dict:
    """Compute per-point log p(y_true | x) in normalised space."""
    theta_use = np.tile(np.asarray(theta, dtype=np.float32), (len(M), 1))
    n_halo = len(M)
    n_r = r_bins.shape[1]

    log_m = np.log10(np.clip(M, 1e10, None)).astype(np.float32)
    log_r = np.log10(np.clip(r_bins, 1e-4, None)).astype(np.float32)

    x_raw = np.zeros((n_halo, n_r, x_dim), dtype=np.float32)
    x_raw[..., 0] = log_m[:, None]
    x_raw[..., 1] = log_r
    x_raw[..., theta_start_idx:theta_start_idx + theta_dim] = theta_use[:, None, :]
    if use_continuous_redshift and redshift_feature_idx < x_dim:
        x_raw[..., redshift_feature_idx] = float(redshift)
    x_flat = x_raw.reshape(-1, x_dim)

    # Build y in the same space the NF was trained on:
    # If mean prior is used: y_target = log10(physical) - mean_model(x)
    # Then normalise: y_norm = (y_target - y_mean) / y_std

    # First, get y in log10 space for log-channels, raw for others
    y_values = np.zeros((n_halo, n_r, y_dim), dtype=np.float32)
    for i, tname in enumerate(target_names):
        if tname in true_profiles:
            vals = true_profiles[tname].astype(np.float32)
            if tname in ALL_PROFILE_LOG_TARGETS:
                vals = np.log10(np.clip(vals, 1e-30, None))
            y_values[:, :, i] = vals

    y_flat = y_values.reshape(-1, y_dim)

    # Subtract mean model if present
    if mean_model is not None:
        x_raw_t = torch.tensor(x_flat, dtype=torch.float32, device=DEVICE)
        BATCH = 8192
        mm_parts = []
        for i in range(0, x_raw_t.shape[0], BATCH):
            mm_pred = predict_mean_from_raw_x(x_raw_t[i:i+BATCH], mean_model)
            mm_parts.append(mm_pred.cpu().numpy())
        mm_contrib = np.concatenate(mm_parts, axis=0)
        y_flat = y_flat - mm_contrib

    # Normalise
    y_norm = (y_flat - y_mean.cpu().numpy()) / y_std.cpu().numpy()
    x_norm = (x_flat - x_mean.cpu().numpy()) / x_std.cpu().numpy()

    x_t = torch.tensor(x_norm, dtype=torch.float32, device=DEVICE)
    y_t = torch.tensor(y_norm, dtype=torch.float32, device=DEVICE)

    # Evaluate log prob
    all_lp = []
    BATCH = 8192
    for i in range(0, x_t.shape[0], BATCH):
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(DEVICE == 'cuda')):
            lp = nf_model.log_prob(x_t[i:i+BATCH], y_t[i:i+BATCH])
        all_lp.append(lp.float().cpu().numpy())

    lp_flat = np.concatenate(all_lp)
    lp_2d = lp_flat.reshape(n_halo, n_r)

    return {
        'log_prob_per_point': lp_flat,
        'log_prob_per_halo_per_radius': lp_2d,
        'mean_log_prob': float(np.mean(lp_flat)),
    }


# Run on all CV sims
cv_nll_by_snap = {}

for snap in cv_snaps_available:
    z_snap = float(training_redshift_by_snap.get(int(snap), np.nan))
    cvp_snap = cv_predictions_by_snap[snap]
    nll_results = {}

    for tag in sorted(cvp_snap.keys()):
        rec = cvp_snap[tag]
        # Need ALL target fields in true_profiles for proper NLL
        # Only evaluate if we have all channels
        tp_full = {}
        for tname in target_names:
            key = f'{tname}_array'
            fp = Path(rec['file'])
            with np.load(fp) as dat:
                if key in dat:
                    tp_full[tname] = np.asarray(dat[key], dtype=np.float64)
        if len(tp_full) < y_dim:
            continue

        result = nf_logprob_cv(
            theta=theta_fid,
            M=rec['M500c'].astype(np.float32),
            r_bins=rec['rr500'],
            true_profiles=tp_full,
            snapnum=int(snap),
            redshift=z_snap,
        )
        nll_results[tag] = result

    cv_nll_by_snap[snap] = nll_results

# Summary
nll_summary_rows = []
for snap in cv_snaps_available:
    z_snap = training_redshift_by_snap.get(snap, np.nan)
    all_lp = np.concatenate([r['log_prob_per_point'] for r in cv_nll_by_snap[snap].values()])
    nll_summary_rows.append({
        'snapnum': snap,
        'redshift': z_snap,
        'mean_log_prob': float(np.mean(all_lp)),
        'median_log_prob': float(np.median(all_lp)),
        'std_log_prob': float(np.std(all_lp)),
        'n_points': len(all_lp),
    })

nll_df = pd.DataFrame(nll_summary_rows)
print('NF log-likelihood on CV set:')
display(nll_df.round(4))

In [ ]:
# ── NLL distribution by mass and radius ──

DIAG_SNAP_NLL = cv_snaps_available[0]
cvp_nll = cv_predictions_by_snap[DIAG_SNAP_NLL]
nll_snap = cv_nll_by_snap[DIAG_SNAP_NLL]
nll_tags = sorted(nll_snap.keys())

if len(nll_tags) > 0:
    # Per-radius mean log-prob
    all_lp_2d = []
    all_masses_nll = []
    for tag in nll_tags:
        lp2d = nll_snap[tag]['log_prob_per_halo_per_radius']  # (n_halo, n_r)
        mask = cvp_nll[tag]['M500c'] >= CV_MASS_CUT
        if mask.sum() == 0:
            continue
        all_lp_2d.append(lp2d[mask])
        all_masses_nll.append(cvp_nll[tag]['M500c'][mask])

    if all_lp_2d:
        lp_2d_all = np.vstack(all_lp_2d)
        masses_nll_all = np.concatenate(all_masses_nll)

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        # Panel 1: log-prob vs radius
        radial_bins_nll = cvp_nll[nll_tags[0]]['radial_bins']
        med_lp = np.median(lp_2d_all, axis=0)
        lo_lp = np.percentile(lp_2d_all, 16, axis=0)
        hi_lp = np.percentile(lp_2d_all, 84, axis=0)
        ax1.plot(radial_bins_nll, med_lp, 'C0-', lw=2)
        ax1.fill_between(radial_bins_nll, lo_lp, hi_lp, color='C0', alpha=0.2)
        ax1.set_xscale('log')
        ax1.set_xlabel('radius [kpc]')
        ax1.set_ylabel('log p(y|x)')
        ax1.set_title(f'Per-radius log-prob (M >= {CV_MASS_CUT:.0e})')
        ax1.grid(True, alpha=0.3)

        # Panel 2: per-halo mean log-prob vs mass
        halo_mean_lp = np.mean(lp_2d_all, axis=1)
        ax2.scatter(np.log10(masses_nll_all), halo_mean_lp, s=6, alpha=0.4, c='C0', rasterized=True)
        ax2.set_xlabel(r'$\log_{10} M_{500c}$')
        ax2.set_ylabel('mean log p(y|x) per halo')
        ax2.set_title(f'Per-halo mean log-prob vs mass')
        ax2.grid(True, alpha=0.3)

        fig.suptitle(f'NF log-likelihood diagnostics | snap={DIAG_SNAP_NLL}', y=1.02)
        plt.show()

## Summary RMSE Table

Overall per-field RMSE across all snapshots, in both log10 (dex) and physical units.

In [ ]:
# ── Global RMSE summary table ──

eps = 1e-30
summary_rows = []

for snap in cv_snaps_available:
    z_snap = training_redshift_by_snap.get(snap, np.nan)
    cvp_snap = cv_predictions_by_snap[snap]

    for fld in CV_FIELDS:
        log_sq, phys_sq = [], []
        for tag in sorted(cvp_snap.keys()):
            rec = cvp_snap[tag]
            mask = rec['M500c'] >= CV_MASS_CUT
            if mask.sum() == 0:
                continue
            true = rec['true_profiles'][fld][mask]
            pred = rec['pred_mu'][fld][mask]
            log_sq.append((np.log10(np.clip(pred, eps, None))
                           - np.log10(np.clip(true, eps, None))) ** 2)
            phys_sq.append((pred - true) ** 2)

        if log_sq:
            summary_rows.append({
                'snapnum': snap,
                'redshift': z_snap,
                'field': fld,
                'RMSE_log10_dex': float(np.sqrt(np.mean(np.concatenate(log_sq)))),
                'RMSE_physical': float(np.sqrt(np.mean(np.concatenate(phys_sq)))),
                'n_halos': sum(1 for t in cvp_snap.values() if (t['M500c'] >= CV_MASS_CUT).any()),
            })

summary_df = pd.DataFrame(summary_rows)
print(f'NF CV RMSE Summary (M500c >= {CV_MASS_CUT:.0e}):')
display(summary_df.round(4))

# Pivot for easy comparison
if len(cv_snaps_available) > 1:
    pivot = summary_df.pivot_table(
        index='field', columns='snapnum', values='RMSE_log10_dex',
    )
    print('\nRMSE [dex] by field and snapshot:')
    display(pivot.round(4))

In [ ]:
# ── Mean model accuracy comparison (if mean prior was used) ──

if mean_model is not None:
    mm_resid_by_field = {fld: [] for fld in DIAG_FIELDS}

    for tag in tags:
        rec = cvp[tag]
        mass = rec['M500c']
        mask = mass >= CV_MASS_CUT
        if mask.sum() == 0:
            continue

        rr500 = rec['rr500'][mask]
        m_masked = mass[mask]
        n_h, nr = rr500.shape

        log_m = np.log10(np.clip(m_masked, 1e10, None)).astype(np.float32)
        log_r = np.log10(np.clip(rr500, 1e-4, None)).astype(np.float32)

        log_m_t = torch.from_numpy(log_m).to(DEVICE)
        log_r_t = torch.from_numpy(log_r).to(DEVICE)
        log_m_flat = log_m_t[:, None].expand(n_h, nr).reshape(-1)
        log_r_flat = log_r_t.reshape(-1)

        z_flat = None
        if mean_model.use_redshift:
            z_val = float(training_redshift_by_snap.get(DIAG_SNAP, 0.0))
            z_flat = torch.full_like(log_m_flat, z_val)

        with torch.no_grad():
            mean_pred_log10 = mean_model(log_m_flat, log_r_flat, redshift=z_flat)
        mean_pred_log10 = mean_pred_log10.cpu().numpy().reshape(n_h, nr, -1)

        for fld in DIAG_FIELDS:
            if fld not in target_names:
                continue
            fi = target_names.index(fld)
            true_log10 = np.log10(np.clip(rec['true_profiles'][fld][mask], eps, None))
            mean_log10 = mean_pred_log10[:, :, fi]
            mm_resid_by_field[fld].append(mean_log10 - true_log10)

    fig, axes = plt.subplots(1, len(DIAG_FIELDS), figsize=(5 * len(DIAG_FIELDS), 4),
                              constrained_layout=True)
    if len(DIAG_FIELDS) == 1:
        axes = [axes]

    for col, fld in enumerate(DIAG_FIELDS):
        if len(mm_resid_by_field[fld]) == 0:
            axes[col].text(0.5, 0.5, 'No data', ha='center', va='center',
                           transform=axes[col].transAxes)
            continue
        all_resid = np.vstack(mm_resid_by_field[fld])
        bias = np.mean(all_resid, axis=0)
        rmse = np.sqrt(np.mean(all_resid ** 2, axis=0))

        ax = axes[col]
        ax.plot(radial_bins, rmse, 'C0-', lw=2, label='RMSE')
        ax.plot(radial_bins, np.abs(bias), 'C3--', lw=1.5, label='|bias|')
        ax.axvline(radial_bins[core_boundary], ls=':', color='red', alpha=0.6)
        ax.set_xscale('log')
        ax.set_ylabel('error [dex]'); ax.set_xlabel('radius [kpc]')
        ax.set_title(f'Mean model: {fld}')
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

        rmse_core = np.sqrt(np.mean(all_resid[:, :core_boundary] ** 2))
        rmse_outer = np.sqrt(np.mean(all_resid[:, core_boundary:] ** 2))
        print(f'Mean model {fld:15s}  RMSE_core={rmse_core:.4f}  RMSE_outer={rmse_outer:.4f}  '
              f'ratio={rmse_core/max(rmse_outer,1e-10):.2f}x')

    fig.suptitle('Mean model (MLP prior) accuracy per radius', y=1.02)
    plt.show()
else:
    print('No mean model — skipping mean model diagnostics.')